## 3D 모델 시각화

In [1]:
import os, random, glob
import open3d as o3d
from pathlib import Path
import numpy as np

# OCC 라이브러리 (STEP 파싱 및 Mesh 변환용)
from OCC.Core.STEPControl import STEPControl_Reader
from OCC.Core.StlAPI import StlAPI_Writer

# OCC 토폴로지 탐색 및 수학적 곡선 추출 모듈
from OCC.Core.TopExp import TopExp_Explorer
from OCC.Core.TopAbs import TopAbs_EDGE, TopAbs_FACE, TopAbs_REVERSED
from OCC.Core.TopLoc import TopLoc_Location
from OCC.Core.GCPnts import GCPnts_QuasiUniformDeflection

# OCC 토폴로지 및 기하 속성 판별 모듈
from OCC.Core.BRep import BRep_Tool
from OCC.Core.BRepAdaptor import BRepAdaptor_Curve, BRepAdaptor_Surface
from OCC.Core.BRepMesh import BRepMesh_IncrementalMesh

# 기하 곡면 타입 전체 임포트
from OCC.Core.GeomAbs import (
    GeomAbs_Plane, GeomAbs_Cylinder, GeomAbs_Cone, 
    GeomAbs_Sphere, GeomAbs_Torus, GeomAbs_BezierSurface, 
    GeomAbs_BSplineSurface, GeomAbs_SurfaceOfRevolution, 
    GeomAbs_SurfaceOfExtrusion, GeomAbs_OffsetSurface, 
    GeomAbs_OtherSurface
)

# 곡면 타입별 색상 지정 (RGB 0~1)
COLOR_MAP = {
    GeomAbs_Plane: [0.8, 0.8, 0.8],               # 평면 (밝은 회색)
    GeomAbs_Cylinder: [0.2, 0.6, 1.0],            # 원기둥 (파란색)
    GeomAbs_Cone: [1.0, 0.8, 0.0],                # 원뿔 (노란색)
    GeomAbs_Sphere: [0.2, 0.8, 0.2],              # 구 (초록색)
    GeomAbs_Torus: [1.0, 0.5, 0.0],               # 토러스/도넛형 (주황색)
    GeomAbs_BSplineSurface: [1.0, 0.2, 0.2],      # B-스플라인 (빨간색)
    GeomAbs_SurfaceOfRevolution: [0.6, 0.2, 0.8], # 회전체 (보라색)
    GeomAbs_SurfaceOfExtrusion: [0.4, 0.8, 0.8],  # 돌출체 (청록색)
    GeomAbs_BezierSurface: [1.0, 0.4, 0.7],       # 베지어 곡면 (분홍색)
    GeomAbs_OffsetSurface: [0.5, 0.5, 0.0],       # 오프셋 곡면 (올리브색)
    GeomAbs_OtherSurface: [0.3, 0.3, 0.3]         # 기타/복합 곡면 (아주 짙은 회색)
}
# 예외처리 색상
DEFAULT_COLOR = [0.0, 0.0, 0.0] # 완전 검은색 (에러)

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
# 경로 설정
base_dir = "./abc_dataset_filtered-1"
prefix_folder_ind = 1 # 0~99
prefix_folder = str(prefix_folder_ind).zfill(4)
prefix_path = Path(base_dir) / prefix_folder

model_id_list = os.listdir(prefix_path) # 00xx0000~00xx9999 
# model_id = random.choice(model_id_list)
model_id = prefix_folder + '0113'

# 경로 조합
model_path = Path(prefix_path) / model_id
step_path = list(model_path.glob('*.step'))[0]

# Open3D로 넘겨주기 위한 브릿지(임시) 파일 경로
temp_stl_path = Path("./_temp_visualization.stl")

print(f"로드할 STEP 파일: {step_path}")

로드할 STEP 파일: abc_dataset_filtered-1\0001\00010113\00010113_a35af5bb7a7a42fe9539f3b7_step_005.step


### 1. 3D Body

In [3]:
# 1. STEP 파일 읽기
reader = STEPControl_Reader()
status = reader.ReadFile(str(step_path))

if status == 1:
    reader.TransferRoots()
    shape = reader.OneShape()
else:
    print("STEP read failure")

# 2. B-Rep 형상을 폴리곤 Mesh 형태로 변환 (Tessellation)
mesh = BRepMesh_IncrementalMesh(shape, 0.1) # 선형 편차(Linear deflection), 값이 작을수록 정밀 분할
mesh.Perform()

# 3. 임시 STL 파일로 저장 (바이너리 모드로 고속 저장)
stl_writer = StlAPI_Writer()
stl_writer.SetASCIIMode(False) 
stl_writer.Write(shape, str(temp_stl_path))

print(f"Mesh 변환 / 임시 STL 파일 생성 완료: {temp_stl_path}")

print("Open3D 뷰어 실행...")

# 1. Open3D로 메쉬 로드
mesh_o3d = o3d.io.read_triangle_mesh(str(temp_stl_path))

# 2. 빛 반사를 계산하기 위한 법선(Normal) 벡터 생성 및 색상 칠하기
mesh_o3d.compute_vertex_normals()
mesh_o3d.paint_uniform_color([0.5, 0.5, 0.5]) # 깔끔한 밝은 회색으로 설정

# 3. 시각화 창 띄우기 (마우스 좌클릭: 회전, 우클릭: 패닝, 휠: 확대/축소)
o3d.visualization.draw_geometries([mesh_o3d], window_name=f"ABC Model Viewer: {model_id}")

# 4. 시각화가 끝나면 임시 생성했던 STL 파일 삭제
if temp_stl_path.exists():
    temp_stl_path.unlink()
    print("시각화 종료 / 임시 파일 정리 완료.")

Mesh 변환 / 임시 STL 파일 생성 완료: _temp_visualization.stl
Open3D 뷰어 실행...
시각화 종료 / 임시 파일 정리 완료.


### 2. B-Rep Edge: WireFrame

In [4]:
# 1. STEP 파일 읽기
reader = STEPControl_Reader()
reader.ReadFile(str(step_path))
reader.TransferRoots()
shape = reader.OneShape()

points = []
lines = []
point_idx = 0

# 2. B-Rep 구조에서 모서리(EDGE)만 탐색
explorer = TopExp_Explorer(shape, TopAbs_EDGE)

while explorer.More():
    edge = explorer.Current()
    
    # Edge의 수학적 기하 곡선(Curve) 정보 가져오기
    adaptor = BRepAdaptor_Curve(edge)
    
    # 곡선을 지정한 편차(deflection) 기준으로 점으로 이산화(Sampling)
    # 두 번째 인자(0.01)가 작을수록 원이나 곡선이 부드럽게 쪼개짐
    discretizer = GCPnts_QuasiUniformDeflection(adaptor, 0.01)
    
    if discretizer.IsDone():
        num_points = discretizer.NbPoints()
        edge_pts = []
        
        # 각 점의 3D 좌표 추출
        for i in range(1, num_points + 1):
            p = discretizer.Value(i) 
            edge_pts.append([p.X(), p.Y(), p.Z()])
            
        # 추출한 점들을 선분(Line) 인덱스로 연결
        if len(edge_pts) > 1:
            points.extend(edge_pts)
            for i in range(len(edge_pts) - 1):
                lines.append([point_idx + i, point_idx + i + 1])
            point_idx += len(edge_pts)
            
    explorer.Next()

print(f"총 {len(lines)}개의 선분(Edge segment) 데이터 추출 완료!")

총 1274개의 선분(Edge segment) 데이터 추출 완료!


In [5]:
print("Edge 형상 WireFrame 뷰어")

# 1. Open3D LineSet 객체 생성
line_set = o3d.geometry.LineSet()

# 2. 점 좌표와 선분 연결 정보 입력
line_set.points = o3d.utility.Vector3dVector(np.array(points, dtype=np.float64))
line_set.lines = o3d.utility.Vector2iVector(np.array(lines, dtype=np.int32))

# 3. 선의 색상 지정 (예: 뚜렷한 파란색 윤곽선)
# RGB 값을 0~1 사이로 입력
colors = [[0.0, 0.4, 1.0] for _ in range(len(lines))]
line_set.colors = o3d.utility.Vector3dVector(colors)

# 4. 시각화 창 띄우기 (창 배경을 흰색으로 바꾸면 2D 도면처럼 보임)
vis = o3d.visualization.Visualizer()
vis.create_window(window_name=f"2D Wireframe Viewer: {model_id}")

# 뷰어 옵션 설정 (배경색 흰색, 선 두께 조절)
opt = vis.get_render_option()
opt.background_color = np.asarray([1.0, 1.0, 1.0])
opt.line_width = 3.0  # 선 두께 (일부 OS에서는 고정일 수 있음)

vis.add_geometry(line_set)
vis.run()
vis.destroy_window()
print("시각화 종료.")

Edge 형상 WireFrame 뷰어
시각화 종료.


### 3. B-Rep Face (CAD SW style)

In [6]:
# 1. STEP 파일 읽기
reader = STEPControl_Reader()
reader.ReadFile(str(step_path))
reader.TransferRoots()
shape = reader.OneShape()

# 2. 전체 형상에 대해 Tessellation 수행
BRepMesh_IncrementalMesh(shape, 0.05)

# === A. Face 메쉬 생성 (색상 속성 부여) ===
combined_mesh = o3d.geometry.TriangleMesh()
explorer_face = TopExp_Explorer(shape, TopAbs_FACE)
face_count = 0

while explorer_face.More():
    face = explorer_face.Current()
    loc = TopLoc_Location()
    triangulation = BRep_Tool.Triangulation(face, loc)
    
    if triangulation is not None:
        adaptor = BRepAdaptor_Surface(face)
        surf_type = adaptor.GetType()
        face_color = COLOR_MAP.get(surf_type, DEFAULT_COLOR)
        
        vertices = []
        for i in range(1, triangulation.NbNodes() + 1):
            p = triangulation.Node(i)
            p.Transform(loc.Transformation())
            vertices.append([p.X(), p.Y(), p.Z()])
            
        triangles = []
        for i in range(1, triangulation.NbTriangles() + 1):
            t = triangulation.Triangle(i)
            n1, n2, n3 = t.Get()
            if face.Orientation() == TopAbs_REVERSED:
                triangles.append([n1-1, n3-1, n2-1])
            else:
                triangles.append([n1-1, n2-1, n3-1])
                
        face_mesh = o3d.geometry.TriangleMesh()
        face_mesh.vertices = o3d.utility.Vector3dVector(np.array(vertices, dtype=np.float64))
        face_mesh.triangles = o3d.utility.Vector3iVector(np.array(triangles, dtype=np.int32))
        face_mesh.paint_uniform_color(face_color)
        
        combined_mesh += face_mesh
        face_count += 1
        
    explorer_face.Next()

# === B. Edge 선분(Wireframe) 추출 ===
edge_points = []
edge_lines = []
point_idx = 0

explorer_edge = TopExp_Explorer(shape, TopAbs_EDGE)
edge_count = 0

while explorer_edge.More():
    edge = explorer_edge.Current()
    adaptor = BRepAdaptor_Curve(edge)
    discretizer = GCPnts_QuasiUniformDeflection(adaptor, 0.01)
    
    if discretizer.IsDone():
        num_points = discretizer.NbPoints()
        pts = []
        
        for i in range(1, num_points + 1):
            p = discretizer.Value(i) 
            pts.append([p.X(), p.Y(), p.Z()])
            
        if len(pts) > 1:
            edge_points.extend(pts)
            for i in range(len(pts) - 1):
                edge_lines.append([point_idx + i, point_idx + i + 1])
            point_idx += len(pts)
            edge_count += 1
            
    explorer_edge.Next()

# Open3D LineSet 객체로 만들기
wireframe = o3d.geometry.LineSet()
wireframe.points = o3d.utility.Vector3dVector(np.array(edge_points, dtype=np.float64))
wireframe.lines = o3d.utility.Vector2iVector(np.array(edge_lines, dtype=np.int32))
# 와이어프레임 색상을 검은색으로 지정
wireframe.paint_uniform_color([0.0, 0.0, 0.0])

print(f"Face: {face_count}개 색상 부여 완료")
print(f"Edge: {edge_count}개 모서리 선분 추출 완료")

Face: 67개 색상 부여 완료
Edge: 378개 모서리 선분 추출 완료


In [8]:
print("CAD-style 렌더링 시작...")

combined_mesh.compute_vertex_normals()

vis = o3d.visualization.Visualizer()
vis.create_window(window_name=f"CAD Style Viewer: {model_id}")

opt = vis.get_render_option()
opt.background_color = np.asarray([1.0, 1.0, 1.0])

# 주의: 이제 삼각형 와이어프레임은 보이지 않도록 False로 설정합니다.
opt.mesh_show_wireframe = False 

# 메쉬(Face)와 라인셋(Edge Wireframe)을 둘 다 뷰어에 추가합니다.
vis.add_geometry(combined_mesh)
vis.add_geometry(wireframe)

vis.run()
vis.destroy_window()
print("시각화 종료.")

CAD-style 렌더링 시작...
시각화 종료.
